# 02 · Descarga de ERA5-Land (variable externa opcional)
**Tesis:** Modelo de ML basado en series temporales para la predicción de eventos climáticos extremos en la región de Puno con datos del SENAMHI (UNI – Maestría en Ciencia de la Computación). **Autor:** Antonio Arthur Quispe Javier. 

**Requiere:** cuenta gratuita en https://cds.climate.copernicus.eu, aceptar la licencia del dataset `reanalysis-era5-land` y su clave API. **No incluya su clave en el cuaderno compartido.** También requiere coordenadas de las estaciones (pendientes de SENAMHI).

In [ ]:
!pip -q install cdsapi xarray netcdf4

In [ ]:
# Cree el archivo ~/.cdsapirc con SU url y SU clave (no la publique):
# url: https://cds.climate.copernicus.eu/api
# key: <SU-CLAVE-PERSONAL>
import cdsapi, xarray as xr, pandas as pd, os
ESTACIONES = {   # <- completar con coordenadas oficiales (SENAMHI)
    'PUNO': (None, None), 'JULI': (None, None), 'ILAVE': (None, None), 'AYAVIRI': (None, None),
    'MAZOCRUZ': (None, None), 'PAMPAHUTA': (None, None), 'CHUQUIBAMBILLA': (None, None)}
assert all(v[0] is not None for v in ESTACIONES.values()), 'Complete lat/lon de cada estación'

In [ ]:
c = cdsapi.Client()
lats = [v[0] for v in ESTACIONES.values()]; lons = [v[1] for v in ESTACIONES.values()]
area = [max(lats)+0.5, min(lons)-0.5, min(lats)-0.5, max(lons)+0.5]   # N, W, S, E
for year in range(1964, 2027):          # ERA5-Land inicia en 1950; se descarga por año
    f = f'era5land_{year}.nc'
    if os.path.exists(f): continue
    c.retrieve('reanalysis-era5-land', {
        'variable': ['2m_temperature', '2m_dewpoint_temperature', 'total_precipitation'],
        'year': str(year), 'month': [f'{m:02d}' for m in range(1, 13)], 'day': [f'{d:02d}' for d in range(1, 32)],
        'time': [f'{h:02d}:00' for h in range(24)], 'area': area, 'data_format': 'netcdf'}, f)

In [ ]:
# Agregación diaria en el punto más cercano a cada estación (tmin/tmax/tmedia provisionales, precip. total)
import glob
frames = []
for f in sorted(glob.glob('era5land_*.nc')):
    ds = xr.open_dataset(f)
    for k, (la, lo) in ESTACIONES.items():
        p = ds.sel(latitude=la, longitude=lo, method='nearest')
        t = p['t2m'] - 273.15
        d = pd.DataFrame({'tmin_era5': t.resample(valid_time='1D').min().to_series(), 'tmax_era5': t.resample(valid_time='1D').max().to_series(),
                          'tmed_era5': t.resample(valid_time='1D').mean().to_series()})
        d['estacion'] = k; frames.append(d)
era = pd.concat(frames); era.to_csv('era5_diario_estaciones.csv'); era.head()

**Advertencia:** ERA5-Land es un reanálisis (resolución ≈ 9 km); no sustituye la observación de estación. Úselo como covariable y para tmin provisional, indicando esta limitación en la tesis.